<a href="https://colab.research.google.com/github/24071a6236-jpg/polymathai/blob/main/Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datasets import load_dataset

toniot = load_dataset("codymlewis/TON_IoT_network")

train_df = toniot["train"].to_pandas()
test_df = toniot["test"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

README.md:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

train_test_network.csv: reconstructing file:   0%|          |  0.00B / 29.9MB            

train_test_network.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/211043 [00:00<?, ? examples/s]

Train shape: (211043, 44)
Test shape : (211043, 44)


In [3]:
train_df
test_df

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211038,192.168.1.32,48286,176.28.50.165,80,tcp,http,65.376610,2665,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211039,192.168.1.32,48288,176.28.50.165,80,tcp,http,65.710346,1987,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211040,192.168.1.32,48292,176.28.50.165,80,tcp,http,65.766512,3922,322,S3,...,0,0,-,-,-,-,-,-,1,xss
211041,192.168.1.32,48294,176.28.50.165,80,tcp,http,65.753940,2401,322,S3,...,0,0,-,-,-,-,-,-,1,xss


In [4]:
# Check data types and separate columns

print("Categorical columns:")
print(train_df.select_dtypes(include=["object"]).columns.tolist())

print("\nNumeric columns:")
print(train_df.select_dtypes(exclude=["object"]).columns.tolist())

Categorical columns:
['src_ip', 'dst_ip', 'proto', 'service', 'conn_state', 'dns_query', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'type']

Numeric columns:
['src_port', 'dst_port', 'duration', 'src_bytes', 'dst_bytes', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'label']


In [5]:
# Target
target = "label"

# Remove target and attack-type information
drop_columns = ["label", "type"]

X_train = train_df.drop(columns=drop_columns)
X_test = test_df.drop(columns=drop_columns)

y_train = train_df[target]
y_test = test_df[target]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (211043, 42)
X_test shape : (211043, 42)
y_train shape: (211043,)
y_test shape : (211043,)


In [6]:
from sklearn.model_selection import train_test_split

X_train_70, X_val_30, y_train_70, y_val_30 = train_test_split(
    X_train,
    y_train,
    test_size=0.30,
    stratify=y_train,
    random_state=42
)

print("70% Training:")
print("X:", X_train_70.shape)
print("y:", y_train_70.shape)

print("\n30% Validation:")
print("X:", X_val_30.shape)
print("y:", y_val_30.shape)

print("\nClass distribution:")
print(y_train_70.value_counts())
print(y_val_30.value_counts())

70% Training:
X: (147730, 42)
y: (147730,)

30% Validation:
X: (63313, 42)
y: (63313,)

Class distribution:
label
1    112730
0     35000
Name: count, dtype: int64
label
1    48313
0    15000
Name: count, dtype: int64


In [7]:
categorical_cols = X_train_70.select_dtypes(include=["object"]).columns.tolist()

print("Number of categorical columns:", len(categorical_cols))
print("\nCategorical columns:")
print(categorical_cols)

print("\nUnique values:")
for col in categorical_cols:
    print(f"{col}: {X_train_70[col].nunique()} unique")

Number of categorical columns: 26

Categorical columns:
['src_ip', 'dst_ip', 'proto', 'service', 'conn_state', 'dns_query', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']

Unique values:
src_ip: 50 unique
dst_ip: 676 unique
proto: 3 unique
service: 9 unique
conn_state: 13 unique
dns_query: 667 unique
dns_AA: 3 unique
dns_RD: 3 unique
dns_RA: 3 unique
dns_rejected: 3 unique
ssl_version: 4 unique
ssl_cipher: 6 unique
ssl_resumed: 3 unique
ssl_established: 3 unique
ssl_subject: 5 unique
ssl_issuer: 4 unique
http_trans_depth: 9 unique
http_method: 4 unique
http_uri: 71 unique
http_version: 2 unique
http_user_agent: 30 unique
http_orig_mime_types: 3 unique
http_resp_mime_types: 10 unique
weird_name: 10 unique
weird_addl: 2 unique


In [8]:
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = X_train_70.select_dtypes(include=["object"]).columns.tolist()

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_70_enc = X_train_70.copy()
X_val_30_enc = X_val_30.copy()

X_train_70_enc[categorical_cols] = encoder.fit_transform(
    X_train_70[categorical_cols]
)

X_val_30_enc[categorical_cols] = encoder.transform(
    X_val_30[categorical_cols]
)

print("Encoded training shape:", X_train_70_enc.shape)
print("Encoded validation shape:", X_val_30_enc.shape)

print("\nRemaining non-numeric columns:")
print(X_train_70_enc.select_dtypes(include=["object"]).columns.tolist())

Encoded training shape: (147730, 42)
Encoded validation shape: (63313, 42)

Remaining non-numeric columns:
[]


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_70_enc)
X_val_scaled = scaler.transform(X_val_30_enc)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)

print("\nTraining mean (first 5 features):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining std (first 5 features):")
print(X_train_scaled.std(axis=0)[:5])

Scaled training shape: (147730, 42)
Scaled validation shape: (63313, 42)

Training mean (first 5 features):
[ 2.83293624e-17  1.05838306e-16  6.92121436e-17  4.62937374e-17
 -9.45835165e-17]

Training std (first 5 features):
[1. 1. 1. 1. 1.]


In [10]:
from sklearn.decomposition import PCA

pca = PCA(n_components=8, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print("PCA training shape:", X_train_pca.shape)
print("PCA validation shape:", X_val_pca.shape)

print("\nExplained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nTotal explained variance:")
print(pca.explained_variance_ratio_.sum())

PCA training shape: (147730, 8)
PCA validation shape: (63313, 8)

Explained variance ratio:
[0.14162686 0.13825188 0.09006847 0.05568143 0.0509491  0.0455914
 0.04399863 0.04331671]

Total explained variance:
0.6094844777970089


In [11]:
from sklearn.model_selection import train_test_split

SEEDS = [42, 43, 44]

seed_splits = {}

for seed in SEEDS:
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train,
        y_train,
        test_size=0.30,
        stratify=y_train,
        random_state=seed
    )

    seed_splits[seed] = {
        "X_train": X_tr,
        "X_val": X_va,
        "y_train": y_tr,
        "y_val": y_va
    }

    print(f"Seed {seed}:")
    print("  Train:", X_tr.shape)
    print("  Val  :", X_va.shape)

Seed 42:
  Train: (147730, 42)
  Val  : (63313, 42)
Seed 43:
  Train: (147730, 42)
  Val  : (63313, 42)
Seed 44:
  Train: (147730, 42)
  Val  : (63313, 42)


In [12]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

SEEDS = [42, 43, 44]

processed = {}

for seed in SEEDS:

    # 1. 70/30 split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.30,
        stratify=y_train,
        random_state=seed
    )

    # 2. Categorical encoding
    categorical_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_tr_enc = X_tr.copy()
    X_val_enc = X_val.copy()

    X_tr_enc[categorical_cols] = encoder.fit_transform(
        X_tr[categorical_cols]
    )

    X_val_enc[categorical_cols] = encoder.transform(
        X_val[categorical_cols]
    )

    # 3. Scaling
    scaler = StandardScaler()

    X_tr_scaled = scaler.fit_transform(X_tr_enc)
    X_val_scaled = scaler.transform(X_val_enc)

    # 4. PCA → 8 components
    pca = PCA(n_components=8)

    X_tr_pca = pca.fit_transform(X_tr_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    # Save everything for this seed
    processed[seed] = {
        "X_train": X_tr_pca,
        "X_val": X_val_pca,
        "y_train": y_tr.to_numpy(),
        "y_val": y_val.to_numpy(),
        "encoder": encoder,
        "scaler": scaler,
        "pca": pca
    }

    print(f"Seed {seed}:")
    print("  Train:", X_tr_pca.shape)
    print("  Val  :", X_val_pca.shape)
    print("  PCA variance:", round(pca.explained_variance_ratio_.sum(), 4))
    print()

Seed 42:
  Train: (147730, 8)
  Val  : (63313, 8)
  PCA variance: 0.6095

Seed 43:
  Train: (147730, 8)
  Val  : (63313, 8)
  PCA variance: 0.6042

Seed 44:
  Train: (147730, 8)
  Val  : (63313, 8)
  PCA variance: 0.6068

